In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Ptc.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 1000,2018-03-01 10:55:17,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 1000,2018-03-01 10:55:21,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,request for payment 1000,2018-03-01 11:34:16,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,2335.0
3,request for payment 1000,2018-03-01 11:34:23,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,7.0
4,request for payment 1000,2018-03-01 15:01:48,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,12445.0
5,request for payment 1000,2018-03-05 14:49:53,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,344885.0
6,request for payment 1000,2018-03-06 10:13:29,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request Payment,SYSTEM,UNDEFINED,69816.0
7,request for payment 1000,2018-03-08 17:31:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Payment Handled,SYSTEM,UNDEFINED,199051.0
8,request for payment 10043,2018-02-20 13:53:11,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
9,request for payment 10043,2018-02-20 13:53:14,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [6.00, 362277.00]                        61266.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [64.68, 1661.05]                         338.4588   quantile_derived    
case:Permit RequestedBudget    continuous     case     yes    [130.85, 4066.04]                        769

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,case:Activity,case:OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 11655,0,False,activity 505,organizational unit 65454,500.867249,125.216812,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 11655,1,False,activity 505,organizational unit 65454,500.867249,125.216812,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,2.0
2,request for payment 11655,2,False,activity 505,organizational unit 65454,500.867249,125.216812,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,60602.0
3,request for payment 11655,3,False,activity 505,organizational unit 65454,500.867249,125.216812,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,1825920.0
4,request for payment 11655,4,False,activity 505,organizational unit 65454,500.867249,125.216812,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,111.0


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Ptc-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0018 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0011 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0009 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0006 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0006 | LR: 1.00e-06
Time taken for scenario model (training): 669.174780 seconds
Time taken for scenario model (validation): 0.449063 seconds
Val loss: {'loss': 0.021194258674742176, 'accuracy': 0.9978865557676575, 'f1_macro': 0.9973291249023778, 'f1_weighted': 0.9978869716987613}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
# scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()